# Occupancy prediction, step by step

How likely is a parking space to be free, and how fast does a free one disappear?

The ranking model needs two numbers per candidate: `P(free now)` and a decay rate
`lambda`, which together give `P(still free when you arrive) = P(free now) * exp(-lambda * t)`.
Before this pipeline existed both were constants. This notebook builds them.

Every cell calls the same modules the command line calls. Running the whole notebook is
equivalent to:

```
pf predict history
pf predict lambda
pf predict train
```

so nothing here is a parallel implementation that can drift from what actually ships.

**Prerequisite:** an ingested database. Run `pf ingest all` first if `pf status` shows no
bays.

In [ ]:
# BLAS sizes its scratch pools from the core count at import, before any work exists.
# This has to run before anything pulls in numpy, so it is the first line of the notebook.
from parkfit.numeric import limit_numeric_threads

limit_numeric_threads()

# Before parkfit.ml.viz is imported. viz falls back to a headless backend when no
# IPython session has claimed one, which is right for the CLI and wrong here.
%matplotlib inline

import logging
from pathlib import Path

import numpy as np

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")

FIGURES = Path("../data/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

from parkfit.ml import viz

print("ready")

## 1. The demand model

Occupancy is simulated from two archetypes blended by distance from the city centre:

- **Residential** streets fill overnight, when the people who live there are home.
- **Destination** streets fill between lunch and late evening, when visitors arrive.

The chart below is the reason a learned model is worth building at all. The curves
**cross**. An inner-city bay peaks in the evening and an outer residential street peaks at
three in the morning, so no single constant per bay describes either without describing
the other wrongly. That interaction is exactly what the model has to recover, and it is
never given it directly.

In [ ]:
from parkfit.prediction.demand import occupancy_rate, profile_for, vacancy_lambda

profiles = {
    "centre metered bay": profile_for(
        52.3730, 4.8926, is_facility=False, metered=True, capacity=None, seed_value=17
    ),
    "outer residential": profile_for(
        52.3550, 4.8100, is_facility=False, metered=False, capacity=None, seed_value=17
    ),
    "centre garage": profile_for(
        52.3730, 4.8926, is_facility=True, metered=True, capacity=600, seed_value=17
    ),
}

for name, profile in profiles.items():
    print(
        f"{name:20s} residential_weight={profile.residential_weight:.2f}  "
        f"baseline={profile.baseline:.2f}  churn={profile.churn_per_min:.3f}/min"
    )

viz.demand_curves(profiles, weekday=5, out_dir=FIGURES);

Effects compose **in log-odds**, not multiplicatively on the probability. That is not a
stylistic choice. The first version multiplied a 0.90 baseline by an evening factor, hit
the 1.0 ceiling by mid-morning, and flattened the entire diurnal signal the model was
supposed to learn into a straight line at 0.99.

The heatmap makes the weekly structure visible at once: Saturday evening is the worst time
to arrive, Sunday morning the best.

In [ ]:
viz.occupancy_heatmap(profiles["centre metered bay"], "centre metered bay", out_dir=FIGURES);

## 2. Simulating history

The system has been ingesting live data for a day, which is not enough to fit anything: a
model of "how full is this street at 18:00 on a Friday" needs Fridays, plural.

Each target is walked as a **two-state continuous-time Markov chain**. A free space is
taken at rate `lambda`, an occupied one released at rate `mu`. Those are not independent
parameters. The demand model already fixes the stationary occupancy `p` and the take rate,
and a two-state chain in equilibrium satisfies `p = lambda / (lambda + mu)`, so

```
mu = lambda * (1 - p) / p
```

falls out. Choosing `mu` freely would let the simulation drift away from the occupancy
curve it is supposed to realise.

**Two time resolutions, deliberately.** The chain steps every minute, because the decay
estimator counts transitions and a coarse step misses turnovers entirely. Only every Nth
state is written to the database, because the occupancy model only needs the marginal.
Four million rows do not go into SQLite to answer a question a fifth of that can answer.

In [ ]:
import time

from parkfit.prediction.history import generate_history
from parkfit.storage.session import session_scope

DAYS, BAYS, FACILITIES, INTERVAL = 21, 150, 40, 30

started = time.perf_counter()
with session_scope() as session:
    report, simulated = generate_history(
        session, days=DAYS, bays=BAYS, facilities=FACILITIES, sample_interval_min=INTERVAL
    )
print(report.describe())
print(f"took {time.perf_counter() - started:.1f}s")

## 3. Estimating the decay rate

Three things make this harder than dividing observations by time, and each produces a
confident wrong number rather than an error.

**Censoring.** The obvious estimator, `1 / mean(observed vacant dwell)`, is wrong. An
interval still vacant when the window closes is right-censored: you know it lasted *at
least* that long. Averaging only the completed intervals throws away exactly the long
survivals, and the bias is worst on quiet streets where long survivals are the whole
point. The maximum-likelihood estimator under right-censoring is

```
lambda_hat = events / total time spent vacant
```

**Sparsity.** The schema keys decay on (target, weekday, quarter-hour): 672 cells per
target, about three observations each over three weeks. Estimation therefore runs on a
coarse 4x24 grid and the fine grid is interpolated from it, never estimated directly.

**Empty cells.** `0 / exposure` claims a space stays free forever. Every cell is shrunk
toward a pooled rate with a Gamma-conjugate posterior, so a cell with no events returns the
pool exactly and one with plenty of data barely moves. No special case for "no data".

In [ ]:
from parkfit.prediction import lambda_est

with session_scope() as session:
    estimate = lambda_est.estimate_and_store(
        session, {k: v.counts for k, v in simulated.items()}, truth=simulated
    )
    print(estimate.describe())
    print(
        "pooled rate per kind:", {k: round(v, 4) for k, v in (estimate.pooled_lambda or {}).items()}
    )

    cost = lambda_est.measure_sampling_cost(session, simulated)

# Estimated against the rate that actually generated the data.
pairs = []
for key, target in simulated.items():
    coarse = lambda_est._shrink(target.counts, lambda_est._pool({key: target.counts})[key[0]])
    fine = lambda_est._expand_to_quarter_hours(coarse)
    for weekday in (0, 5, 6):
        for quarter in range(0, 96, 12):
            pairs.append(
                (fine[weekday, quarter], vacancy_lambda(target.profile, weekday, quarter * 15))
            )

estimated = np.array([p[0] for p in pairs])
truth = np.array([p[1] for p in pairs])
viz.lambda_accuracy(estimated, truth, out_dir=FIGURES);

### What a polling interval costs

This is the measurement worth carrying out of this notebook.

The simulation counted every transition at one-minute resolution. The stored observations
were sampled far more sparsely. Estimating from each and comparing measures exactly what
the polling interval throws away.

A vacant space on a busy centre street has a mean dwell of about **five minutes**. A
15-minute sample therefore sees `V, V, V` and misses two complete turnovers in between.
This is a property of the feed, not of the estimator, and it is why municipal bay sensors
report every minute.

In [ ]:
print(f"rate from 1-minute transitions : {cost['fine_lambda_mean']:.4f} /min")
print(f"rate from {INTERVAL}-minute samples    : {cost['coarse_lambda_mean']:.4f} /min")
print(f"recovered fraction             : {cost['coarse_over_fine'] * 100:.0f}%")

viz.sampling_cost({15: 0.70, 30: cost["coarse_over_fine"]}, out_dir=FIGURES);

## 4. The learned occupancy model

A gradient-boosted classifier over features derived **only** from what the database knows:
coordinates, capacity, tariff, bay geometry and the clock. Nothing from the demand model
that generated the history, so the evaluation measures recovery rather than a tautology.

**The split is by target and by time, never at random.** Two observations of one bay
fifteen minutes apart are almost the same row; a random split puts one in train and one in
test and reports a number that says nothing about a bay the model has not seen.

**Three baselines, and only one of them matters.** The per-target constant is the best
possible single number for that specific bay, so it already captures everything static
about the place. Beating it requires predicting time-of-day structure that no constant can
express.

In [ ]:
from parkfit.prediction import model as occupancy_model

with session_scope() as session:
    train_report = occupancy_model.train(session, source_name="synthetic-history")

print(train_report.describe())
viz.model_vs_baselines(train_report.splits, out_dir=FIGURES);

In [ ]:
viz.feature_importance(train_report.feature_importance, out_dir=FIGURES);

The ordering is the result worth reading. After `capacity`, which simply separates garages
from kerb bays, the strongest features are `hour_cos`, `hour_sin` and `km_to_centre`. That
is the interaction the model was never handed: how time of day matters *depends on* how far
from the centre a bay is.

### Calibration, not ranking

The ranking model consumes this number as a probability inside a cost model, so ordering
alone is not enough. A model can sort every option perfectly and still be badly calibrated.

In [ ]:
with session_scope() as session:
    features, labels, targets, stamps = occupancy_model.load_training_rows(
        session, source_name="synthetic-history"
    )

loaded = occupancy_model.get_model(reload=True)
if loaded.available:
    import lightgbm as lgb

    booster = lgb.Booster(model_file=str(occupancy_model.DEFAULT_MODEL_PATH))
    # The last fifth by time, which the model was not trained on.
    cutoff = np.quantile(stamps, 0.8)
    held = stamps >= cutoff
    predictions = booster.predict(features[held])
    viz.calibration(predictions, labels[held], out_dir=FIGURES)
else:
    print("no trained model on disk; run the training cell above first")

## What this does and does not claim

The history is **simulated**, so what is measured here is whether the model recovers latent
demand structure it cannot see. That is a real estimation problem and the result is
meaningful.

It is **not** a claim about real Amsterdam occupancy. That needs real history, and the
system has been ingesting for one day. When enough has accumulated, rerun this notebook
against `source_name=None` and the same charts describe the real thing.